# 3D Diffusion Model for Scalar Field Theory

This notebook demonstrates how to use the 3D diffusion model for lattice field theory simulations.


In [1]:
import os
import numpy as np
import torch
import pytorch_lightning as pl
import matplotlib.pyplot as plt

from diffusion_lightning_3d import (
    DiffusionModel3D, FieldDataModule3D, 
    grab, get_mag_3d, get_abs_mag_3d, get_chi2_3d, get_UL_3d, mag_3d
)

# Create directories
for folder in ['data', 'figures', 'models']:
    os.makedirs(folder, exist_ok=True)

if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")


Using GPU: NVIDIA GeForce RTX 4090


## 1. Configuration


In [2]:
# Lattice parameters
L = 32          # 3D lattice size: L x L x L (use smaller L for 3D due to memory)
k = 0.5     # Hopping parameter
l = 2.5       # Coupling
sigma = 25.0    # SDE noise scale

# Training parameters
n_epochs = 250
batch_size = 16  # Smaller batch size for 3D due to memory constraints
lr = 1e-3

# Sampling parameters
num_steps = 1000
num_samples = 256  # Fewer samples for 3D

# Data path (adjust to your 3D data)
data_path = f"data/cfgs_k={k}_l={l}_{L}^3_t=10.jld2"


In [3]:
data_module = FieldDataModule3D(
    data_path=data_path,
    batch_size=batch_size,
    normalize=True
)

## 5. Generate Samples


In [21]:
ckdir=f"/home/tyy/DM/DMasSQ-main/models/3D_2/diffusion_L{L}_k{k}_l{l}_{L}^3_t=10-epoch=49.ckpt"

In [22]:
model = DiffusionModel3D.load_from_checkpoint(ckdir,map_location='cuda:3')
checkpoint = torch.load(ckdir, map_location='cpu') # Use 'cpu' if you don't need a GPU

In [7]:
# MAALA Sampling (Metropolis-Adjusted Annealed Langevin Algorithm) for 3D
# 使用 sample_batch_size 分批生成以节省显存
num_samples_mala = 256
sample_batch_size_mala = 128  # 每批生成的样本数，根据显存调整

samples_mala, acceptance_rate = model.sample_mala(
    num_samples=num_samples_mala,
    num_steps=500,
    sample_batch_size=sample_batch_size_mala,  # 分批处理
    k=k,
    l=l,
    mh_last_steps=0,
    alpha=1.0,
)

print(f"\nMAALA: {num_samples_mala} samples, acceptance rate: {acceptance_rate:.4f}")

KeyboardInterrupt: 

In [ ]:
# 可视化 MAALA 3D 样本 (显示 z 方向中间切片)
samples_mala_np = grab(samples_mala[:, 0, :, :, :])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for idx, ax in enumerate(axes.flat):
    if idx < len(samples_mala_np):
        mid_z = samples_mala_np.shape[3] // 2
        im = ax.imshow(samples_mala_np[idx, :, :, mid_z], cmap='RdBu_r')
        ax.set_title(f'MAALA Sample {idx+1}')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle(f'MAALA 3D Samples (acc rate: {acceptance_rate:.3f})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
model = DiffusionModel3D.load_from_checkpoint(ckdir).to(torch.device("cuda:0"))

In [ ]:
checkpoint = torch.load(ckdir, map_location='cpu') # Use 'cpu' if you don't need a GPU


In [ ]:
checkpoint['epoch']

In [23]:
# Generate samples with memory optimization
num_samples_generate = 128
sample_batch_size = 128

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,  # Use fewer steps for quick demo
    sample_batch_size=sample_batch_size,
    show_progress=True,
    # ===== MEMORY OPTIMIZATION FOR SAMPLING =====
    use_amp=False,                # Use AMP during sampling (~30-50% faster, less memory)
)

print(f"Generated samples shape: {samples.shape}")


Sampling: 100%|██████████| 500/500 [03:17<00:00,  2.53it/s]

Generated samples shape: torch.Size([128, 1, 32, 32, 32])


## 6. Visualize Samples


In [ ]:
# Visualize 3D samples using slices
samples_np = grab(samples[:, 0, :, :, :])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for idx, ax in enumerate(axes.flat):
    if idx < len(samples_np):
        # Show middle slice in z-direction
        mid_z = samples_np.shape[3] // 2
        im = ax.imshow(samples_np[idx, :, :, mid_z], cmap='RdBu_r')
        ax.set_title(f'Sample {idx+1} (z={mid_z})')
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Generated 3D Field Configurations (Middle Z-Slice)', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize multiple slices of a single sample
sample_idx = 0
sample = samples_np[sample_idx]

n_slices = 4
slice_indices = np.linspace(0, sample.shape[2]-1, n_slices, dtype=int)

fig, axes = plt.subplots(1, n_slices, figsize=(16, 4))

for i, z_idx in enumerate(slice_indices):
    im = axes[i].imshow(sample[:, :, z_idx], cmap='RdBu_r')
    axes[i].set_title(f'z = {z_idx}')
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046)

plt.suptitle(f'Sample {sample_idx+1}: Z-Slices Through 3D Volume', fontsize=14)
plt.tight_layout()
plt.show()


## 7. Compute Observables


In [8]:
import numpy as np
from scipy.special import binom
import h5py


def norm(x, dtype=np.float64, copy=True):
    """Normalize array to [-1, 1] using min/max."""
    x = np.array(x, dtype=dtype, copy=copy)
    xmin = x.min()
    xmax = x.max()
    scale = xmax - xmin
    if scale == 0:
        x.fill(dtype(0))
        return x
    x -= xmin
    x /= scale
    x -= dtype(0.5)
    x *= dtype(2.0)
    return x


def cumulants_from_moments(moments):
    """Cumulants from moments using the recursion in Eq. (1.221).

    `moments` has shape (order, ...). The output has the same shape.
    """
    m = np.asarray(moments, dtype=np.float64)
    order = m.shape[0]
    kappa = np.empty_like(m, dtype=np.float64)
    kappa[0] = m[0]
    for n in range(2, order + 1):
        val = m[n - 1].copy()
        for i in range(1, n):
            val -= binom(n - 1, i - 1) * kappa[i - 1] * m[n - i - 1]
        kappa[n - 1] = val
    return kappa


def _prepare_site_data(data, dtype=np.float32, lattice_axes=(-3, -2, -1)):
    """Return (x_flat, site_shape) with x_flat shaped (n_samples, n_sites).

    We compute local (single-site) cumulants κ_n(x). Any non-lattice, non-sample
    axes (e.g. a channel axis of length 1) are averaged out first.
    """
    x = np.asarray(data)

    # Drop a singleton channel axis if present: (N,1,L,L,L) -> (N,L,L,L)
    if x.ndim >= 4 and x.shape[1] == 1:
        x = x[:, 0]

    x = np.asarray(x, dtype=dtype)

    nd = x.ndim
    lat_axes = tuple(ax if ax >= 0 else nd + ax for ax in lattice_axes)
    other_axes = tuple(ax for ax in range(1, nd) if ax not in lat_axes)
    if other_axes:
        x = x.mean(axis=other_axes, dtype=np.float64)

    site_shape = x.shape[1:]
    x_flat = x.reshape(x.shape[0], -1)
    return x_flat, site_shape


def lattice_bootstrap_cumulants(
    data,
    order,
    axis=0,
    n_boot=1000,
    seed=None,
    dtype=np.float64,
    lattice_axes=(-3, -2, -1),
    n_bins=200,
    store_dtype=np.float32,
):
    """Bootstrap estimate of local cumulants κ_n for 3D lattices.

    Paper definition (local cumulants):
      1) compute κ_n(x) at each lattice site x from the ensemble,
      2) average over sites: κ_n = (1/V) Σ_x κ_n(x).

    Errors are estimated by bootstrapping over *bins* of configurations.

    Notes (3D): storing per-bin, per-site moments is memory-heavy.
    - default `n_bins=200` keeps memory reasonable for 32^3.
    - moments are stored in `store_dtype` (float32) and accumulated in float64.
    """
    rng = np.random.default_rng(seed)

    x_flat, _ = _prepare_site_data(data, dtype=store_dtype, lattice_axes=lattice_axes)
    n_samples, n_sites = x_flat.shape

    n_bins = int(min(n_bins, n_samples))
    bin_size = n_samples // n_bins
    if bin_size < 1:
        raise ValueError("n_bins is too large for the number of samples")

    n_used = bin_size * n_bins
    if n_used != n_samples:
        x_flat = x_flat[:n_used]
        n_samples = n_used

    # Per-bin, per-site moments: shape (n_bins, order, n_sites)
    per_bin_m = np.empty((n_bins, order, n_sites), dtype=store_dtype)
    for b in range(n_bins):
        xb = x_flat[b * bin_size : (b + 1) * bin_size].astype(np.float64, copy=False)
        x_pow = xb.copy()
        for k in range(order):
            per_bin_m[b, k] = x_pow.mean(axis=0, dtype=np.float64).astype(store_dtype, copy=False)
            if k != order - 1:
                x_pow *= xb

    boot_kappa = np.empty((n_boot, order), dtype=np.float64)
    for b in range(n_boot):
        idx = rng.integers(0, n_bins, size=n_bins, dtype=np.int32)
        m = per_bin_m[idx].mean(axis=0, dtype=np.float64)  # (order, n_sites)
        kappa = cumulants_from_moments(m)  # (order, n_sites)
        boot_kappa[b] = kappa.mean(axis=1, dtype=np.float64)

    cumulants_means = boot_kappa.mean(axis=0)
    boot_errors = boot_kappa.std(axis=0)
    return cumulants_means, boot_errors

In [24]:
# samples_re = grab(samples_mala[:, 0, :, :,:])
samples_re = grab(samples[:, 0, :, :,:])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=2, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=2, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=2, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=2, seed=1)

In [25]:
data_module.setup()

In [26]:
np.save("data/cfgs_dm3_k=0.5_l=2.5_32^3_t=10-epoch=49.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))

In [11]:
np.save("data/cfgs_dm3_mala_k=0.5_l=2.5_64^3_t=10-epoch=499.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))

In [ ]:
np.save("data/cfgs_dm3_k=0.5_l=2.5_64^3_t=10-epoch=499.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))

In [ ]:
ckdir="/home/tyy/DM/DMasSQ-main/models/3D_2/diffusion_L64_k0.5_l2.5_64^3_t=10-epoch=04.ckpt"
model = DiffusionModel3D.load_from_checkpoint(ckdir).to(torch.device("cuda:3"))
# Generate samples with memory optimization
num_samples_generate = 256
sample_batch_size = 16

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,  # Use fewer steps for quick demo
    sample_batch_size=sample_batch_size,
    show_progress=True,
    # ===== MEMORY OPTIMIZATION FOR SAMPLING =====
    use_amp=False,                # Use AMP during sampling (~30-50% faster, less memory)
)

print(f"Generated samples shape: {samples.shape}")
samples_re = grab(samples[:, 0, :, :,:])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=2, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=2, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=2, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=2, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_64^3_t=10-epoch=04.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))


In [ ]:
ckdir="/home/tyy/DM/DMasSQ-main/models/3D_2/diffusion_L64_k0.5_l2.5_64^3_t=10-epoch=09.ckpt"
model = DiffusionModel3D.load_from_checkpoint(ckdir).to(torch.device("cuda:3"))
# Generate samples with memory optimization
num_samples_generate = 256
sample_batch_size = 16

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,  # Use fewer steps for quick demo
    sample_batch_size=sample_batch_size,
    show_progress=True,
    # ===== MEMORY OPTIMIZATION FOR SAMPLING =====
    use_amp=False,                # Use AMP during sampling (~30-50% faster, less memory)
)

print(f"Generated samples shape: {samples.shape}")
samples_re = grab(samples[:, 0, :, :,:])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=2, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=2, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=2, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=2, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_64^3_t=10-epoch=09.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))


In [ ]:
ckdir="/home/tyy/DM/DMasSQ-main/models/3D_2/diffusion_L64_k0.5_l2.5_64^3_t=10-epoch=14.ckpt"
model = DiffusionModel3D.load_from_checkpoint(ckdir).to(torch.device("cuda:3"))
# Generate samples with memory optimization
num_samples_generate = 256
sample_batch_size = 16

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,  # Use fewer steps for quick demo
    sample_batch_size=sample_batch_size,
    show_progress=True,
    # ===== MEMORY OPTIMIZATION FOR SAMPLING =====
    use_amp=False,                # Use AMP during sampling (~30-50% faster, less memory)
)

print(f"Generated samples shape: {samples.shape}")
samples_re = grab(samples[:, 0, :, :,:])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=2, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=2, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=2, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=2, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_64^3_t=10-epoch=14.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))


In [ ]:
ckdir="/home/tyy/DM/DMasSQ-main/models/3D_2/diffusion_L64_k0.5_l2.5_64^3_t=10-epoch=499.ckpt"
model = DiffusionModel3D.load_from_checkpoint(ckdir).to(torch.device("cuda:3"))
# Generate samples with memory optimization
num_samples_generate = 256
sample_batch_size = 16

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,  # Use fewer steps for quick demo
    sample_batch_size=sample_batch_size,
    show_progress=True,
    # ===== MEMORY OPTIMIZATION FOR SAMPLING =====
    use_amp=False,                # Use AMP during sampling (~30-50% faster, less memory)
)

print(f"Generated samples shape: {samples.shape}")
samples_re = grab(samples[:, 0, :, :,:])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=2, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=2, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=2, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=2, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_64^3_t=10-epoch=499.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))


In [ ]:
ckdir="/home/tyy/DM/DMasSQ-main/models/3D_2/diffusion_L64_k0.5_l2.5_64^3_t=10-epoch=249.ckpt"
model = DiffusionModel3D.load_from_checkpoint(ckdir).to(torch.device("cuda:3"))
# Generate samples with memory optimization
num_samples_generate = 64
sample_batch_size = 16

samples = model.sample(
    num_samples=num_samples_generate,
    num_steps=500,  # Use fewer steps for quick demo
    sample_batch_size=sample_batch_size,
    show_progress=True,
    # ===== MEMORY OPTIMIZATION FOR SAMPLING =====
    use_amp=False,                # Use AMP during sampling (~30-50% faster, less memory)
)

print(f"Generated samples shape: {samples.shape}")
samples_re = grab(samples[:, 0, :, :,:])
cfgs_dm = np.array(samples_re,dtype=np.float64,copy=True)
cfgs_dm2=np.array(np.concatenate([samples_re,-samples_re],axis=0),dtype=np.float64,copy=True)
cfgs = norm(np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True))
cumulants_cfgs, errs_cfgs = lattice_bootstrap_cumulants(cfgs, 8, n_boot=2, seed=2)
cumulants_cfgs_dm, errs_cfgs_dm = lattice_bootstrap_cumulants(cfgs_dm, 8, n_boot=2, seed=1)
cumulants_cfgs_dm2, errs_cfgs_dm2 = lattice_bootstrap_cumulants(cfgs_dm2, 8, n_boot=2, seed=1)
cfgs_dm3=cfgs_dm2 * np.sqrt(cumulants_cfgs[1]/cumulants_cfgs_dm2[1])
cumulants_cfgs_dm3, errs_cfgs_dm3 = lattice_bootstrap_cumulants(cfgs_dm3, 8, n_boot=2, seed=1)
data_module.setup()
np.save("data/cfgs_dm3_k=0.5_l=2.5_64^3_t=10-epoch=249.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))


In [ ]:
np.save("data/cfgs_dm3_k=0.5_l=2.5_64^3_t=10.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))

In [ ]:
np.save("data/cfgs_dm3_k=0.5_l=2.5_32^3_t=10.npy", data_module.renorm(cfgs_dm3).transpose(1,2,3,0))

In [15]:
print("Type \t | κ_2\t\t  | κ_4\t\t    | κ_6 \t     | κ_8")
print("    data", end=" |")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion", end="|")
for c, e in zip(cumulants_cfgs_dm[1::2], errs_cfgs_dm[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion2", end="|")
for c, e in zip(cumulants_cfgs_dm2[1::2], errs_cfgs_dm2[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion3", end="|")
for c, e in zip(cumulants_cfgs_dm3[1::2], errs_cfgs_dm3[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")

Type 	 | κ_2		  | κ_4		    | κ_6 	     | κ_8
    data |0.5171 ± 0.0000 |-0.4963 ± 0.0000 |1.9696 ± 0.0004 |-16.6267 ± 0.0048 |
Diffusion|0.5095 ± 0.0006 |-0.4587 ± 0.0013 |1.7166 ± 0.0087 |-13.5074 ± 0.1049 |
Diffusion2|0.5136 ± 0.0005 |-0.4720 ± 0.0012 |1.8198 ± 0.0077 |-14.8888 ± 0.0931 |
Diffusion3|0.5171 ± 0.0006 |-0.4785 ± 0.0012 |1.8574 ± 0.0078 |-15.3006 ± 0.0957 |

In [ ]:
print("Type \t | κ_2\t\t  | κ_4\t\t    | κ_6 \t     | κ_8")
print("    data", end=" |")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion", end="|")
for c, e in zip(cumulants_cfgs_dm[1::2], errs_cfgs_dm[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion2", end="|")
for c, e in zip(cumulants_cfgs_dm2[1::2], errs_cfgs_dm2[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion3", end="|")
for c, e in zip(cumulants_cfgs_dm3[1::2], errs_cfgs_dm3[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")

In [ ]:
print("Type \t | κ_2\t\t  | κ_4\t\t    | κ_6 \t     | κ_8")
print("    data", end=" |")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion", end="|")
for c, e in zip(cumulants_cfgs_dm[1::2], errs_cfgs_dm[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion2", end="|")
for c, e in zip(cumulants_cfgs_dm2[1::2], errs_cfgs_dm2[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion3", end="|")
for c, e in zip(cumulants_cfgs_dm3[1::2], errs_cfgs_dm3[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")

In [ ]:
print("Type \t | κ_2\t\t  | κ_4\t\t    | κ_6 \t     | κ_8")
print("    data", end=" |")
for c, e in zip(cumulants_cfgs[1::2], errs_cfgs[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion", end="|")
for c, e in zip(cumulants_cfgs_dm[1::2], errs_cfgs_dm[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion2", end="|")
for c, e in zip(cumulants_cfgs_dm2[1::2], errs_cfgs_dm2[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")
print()
print("Diffusion3", end="|")
for c, e in zip(cumulants_cfgs_dm3[1::2], errs_cfgs_dm3[1::2]):
	print(f"{c:.4f} ± {e:.4f}", end=" |")

In [ ]:
cfgs.shape

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(cfgs[11,:,:,16])
plt.colorbar()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(cfgs_dm3[11,:,:,16])
plt.colorbar()
plt.show()

In [ ]:
cfgs = np.array(h5py.File(data_path, "r")["cfgs"],dtype=np.float64,copy=True)

In [ ]:
cfgs.shape

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=3, ncols=5, figsize=(8.4, 5),
                        subplot_kw={'xticks': [], 'yticks': []})
for i in range(3):
    for j in range(5):
        axs[i, j].imshow(cfgs[i*5+j,32,:,:])
plt.tight_layout(pad=0.5)
plt.show()

In [ ]:
cfgs_dm = np.load("data/cfgs_dm3_k=0.5_l=2.5_64^3_t=10-epoch=499.npy",mmap_mode="r").transpose(3,0,1,2)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(nrows=3, ncols=5, figsize=(8.4, 5),
                        subplot_kw={'xticks': [], 'yticks': []})
for i in range(3):
    for j in range(5):
        axs[i, j].imshow(cfgs_dm[i*5+j,32,:,:])
plt.tight_layout(pad=0.5)
plt.show()

In [ ]:
from scipy import stats

In [ ]:
k2data=stats.kstat(cfgs, 2,axis=0)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(k2data[:,:,32])
plt.title("$\\kappa_2$ from Data")
plt.colorbar()
plt.show()

In [ ]:
k2dm=stats.kstat(cfgs_dm, 2,axis=0)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(k2dm[:,:,32])
plt.title("$\\kappa_2$ from DM")
plt.colorbar()
plt.show()

In [ ]:
kdata=stats.kstat(cfgs, 4,axis=0)
import matplotlib.pyplot as plt
plt.imshow(kdata[:,:,32])
plt.title("$\\kappa_4$ from Data")
plt.colorbar()
plt.show()
k4dm=stats.kstat(cfgs_dm, 4,axis=0)
import matplotlib.pyplot as plt
plt.imshow(k4dm[:,:,32])
plt.title("$\\kappa_4$ from DM")
plt.colorbar()
plt.show()
